In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors as mcolors

from matplotlib.lines import Line2D


# Seaborn style for the figure
sns.set_style("whitegrid")
sns.set_context("talk")

In [ ]:
# No limit on number of columns
pd.set_option('display.max_columns', None)

# Optional settings to prevent line breaks
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## Functions

In [ ]:
# Allows plotting evaluation in both reconstructed and latent spaces
def plot_metric_grid_all(
        df_full,
        df_only_coreset,
        df_latent,
        df_latent_ft,
        df_MLP=pd.DataFrame(),
        df_TDColer=pd.DataFrame(),
        spaces=('original', 'latent'),
        max_ipc=100,
        metric='test_balanced',
        methods=('k-means', 'k-center', 'ag'),
        models=None,
        x_label='Instances Per Class',
        y_label='Test balanced',
        name_savefig='figure.png'
    ):
    """
    Grid dinámico metric vs IPC:
      • Filas → cada modelo de `models`.
      • Columnas → bloques por espacio (cada bloque = len(methods) columnas).
    Fondo de cada bloque coloreado según espacio (original / latent).
    Dentro de cada celda:
      - Curvas media±std de cada técnica (Original, CorVAE, TDColer).
      - Línea base full-data.
      - Cuadro con valor Full-data y máximo de CorVAE.
    """

    

    # 1) Unir DataFrames con columna 'space' y 'technique'
    df_list = []
    for space in spaces:
        d_latent = df_latent[df_latent['space'] == space]
        d_tdcol  = df_TDColer[df_TDColer['space'] == space] if not df_TDColer.empty else pd.DataFrame()
        for src, tech in [
            (df_only_coreset, 'Original'),
            (d_latent,       'CorVAE'),
            #(df_latent_ft,  'latent_ft'),
            (d_tdcol,        'TDColer'),
        ]:
            if src.empty:
                continue
            tmp = src.copy()
            tmp['technique'] = tech
            tmp['space']     = space
            df_list.append(tmp)

    print("Printing df_list info")
    for i in df_list:
        print(i)

    df_all = pd.concat(df_list, ignore_index=True)
    df_all = df_all[df_all['IPC'] <= max_ipc]

    # 2) Preparo figura y ejes
    n_rows = len(models)
    n_cols = len(spaces) * len(methods)
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(4*n_cols, 3*n_rows),
        sharex=True, sharey=True
    )
    axes = np.array(axes).reshape(n_rows, n_cols)

    # Fixed palette for lines
    tech_colors = {
        'Original': 'C0',
        'CorVAE':   'C2',
        'TDColer':  'C4',
    }
    baseline_color = 'gray'

    # Pastel palette for space backgrounds
    bg_palette = sns.color_palette("pastel", len(spaces))
    bg_colors  = dict(zip(spaces, bg_palette))

    remap1 = {
        'k-means':  'KM',
        'k-center': 'K-Centers',
        'ag':       'AG',
        'lc':      'LC',
    }

    remap2 = {
        'logreg': 'LR',
        'svc':    'SVC',
        'rf':     'RF',
        'xgb':    'XGBoost',
        'naive_bayes':   'NB',
        'mlp':   'MLP',
        'knn':  'KNN',
        
    }

    # 3) Rellenar cada subplot
    for i, model in enumerate(models):
        for k, space in enumerate(spaces):
            for j, method in enumerate(methods):
                col = k * len(methods) + j
                ax = axes[i, col]

                # Background shading according to space
                rgba = mcolors.to_rgba(bg_colors[space], alpha=0.1)
                ax.set_facecolor(rgba)

                sub = df_all[
                    (df_all['model']  == model) &
                    (df_all['method'] == method) &
                    (df_all['space']  == space)
                ]
                
                if sub.empty:
                    ax.set_visible(False)
                    continue

                ax.set_title(f"{remap2[model]} – {remap1[method]}", loc='left', pad=6)

                # Draw curves and calculate CorVAE maximum
                corvae_max_val = None
                corvae_max_ipc = None
                for tech, color in tech_colors.items():
                    sp = sub[sub['technique'] == tech]
                    if sp.empty:
                        
                        continue
                    #print(sp)
                    grp = sp.groupby('IPC')[metric].agg(['mean','std']).reset_index()
                    ax.plot(grp['IPC'], grp['mean'],
                            marker='o', label=tech, color=color)
                    ax.fill_between(
                        grp['IPC'],
                        grp['mean'] - grp['std'],
                        grp['mean'] + grp['std'],
                        alpha=0.2, color=color
                    )
                    # if it is CorVAE, calculate its maximum
                    if tech == 'CorVAE':

                        idx = grp['mean'].idxmax()
                        corvae_max_val = grp.at[idx, 'mean']
                        corvae_max_ipc = grp.at[idx, 'IPC']

                # Full-data baseline
                baseline = df_full[df_full['model'] == model][metric].mean()
                ax.axhline(y=baseline,
                           linestyle='--',
                           color=baseline_color,
                           label='Full-data')

                ax.grid(True, linestyle='--', alpha=0.5)
                ax.set_xlim(0, max_ipc)

    fig.supxlabel(x_label, fontsize=14)
    fig.supylabel(y_label, fontsize=14, rotation='vertical', x=0.02)


    legend_elements = [
        Line2D([0], [0], marker='o', color=tech_colors['Original'], label='Original'),
        Line2D([0], [0], marker='o', color=tech_colors['CorVAE'],   label='CorVAE'),
        Line2D([0], [0], linestyle='--', color=baseline_color,      label='Full-data'),
    ]

    if 'TDColer' in df_all['technique'].unique():
        legend_elements.append(
            Line2D([0], [0], marker='o', color=tech_colors['TDColer'], label='TDColer')
        )

    fig.legend(
        handles=legend_elements,
        loc='upper center',
        ncol=len(legend_elements),
        #title='Technique',
        bbox_to_anchor=(0.5, 1.03),  # sube un poco más la leyenda
        fontsize=12,
        title_fontsize=13
    )

    # Adjust layout to avoid overlapping
    plt.tight_layout(rect=[0, 0, 1, 0.97])  # more space at the top
    plt.savefig(name_savefig, dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# Allows plotting evaluation only in one of the two spaces
def plot_metric_grid(df_random,
                     df_original,
                     df_latent,
                     df_latent_ft,
                     space='original',
                     max_ipc=100,
                     metric='test_balanced',
                     x_label='Instances Per Class',
                     y_label='Test balanced',
                     methods=['k-means',
                              'k-center',
                              'ag'
                              ]):
    """
    Dibuja en una única figura (grid) la curva de `metric` vs IPC
    para cada combinación (método, modelo).  
    Filas → modelos, Columnas → métodos.  
    Dentro de cada celda: las tres técnicas de distilación
    (original, latent, latent_ft) con media ± std, y línea base Full-data.
    """

    df_latent = df_latent[df_latent['space'] == space]
    df_latent_ft = df_latent_ft[df_latent_ft['space'] == space]

    # 1) Concatena solo los tres DataFrames de distilación
    dfs = [
        (df_original.copy(),  'original'),
        (df_latent.copy(),    'latent'),
        (df_latent_ft.copy(), 'latent_ft'),
    ]
    df_list = []
    for df, tech in dfs:
        df['technique'] = tech
        df_list.append(df)
    df_all_init = pd.concat(df_list, ignore_index=True)

    df_all = df_all_init[df_all_init['IPC'] <= max_ipc]

    # 3) Métodos y modelos para el grid
    #methods = sorted([m for m in df_all['method'].unique()
    #                  if m not in ['random', 'Full-data']])
    models  = sorted(df_all['model'].unique())

    # Swap: rows = models, columns = methods
    n_rows, n_cols = len(models), len(methods)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(4*n_cols, 3*n_rows),
                             sharex=True, sharey=True)

    # 4) Paleta para las tres técnicas
    palette = {
        'original':  'C0',
        'latent':    'C2',
        'latent_ft': 'C3',
        'TDColer':   'C4',
    }

    # 5) Rellenar cada celda
    for i, model in enumerate(models):
        for j, method in enumerate(methods):
            ax = axes[i, j] if n_rows > 1 and n_cols > 1 else (
                 axes[j] if n_rows == 1 else axes[i])
            
            

            sub = df_all[
                (df_all['method'] == method) &
                (df_all['model']  == model)
            ]
            if sub.empty:
                ax.set_visible(False)
                continue

            # Internal title of each subplot
            ax.set_title(f"{model} – {method}", loc='left', pad=10)

            # Draw curves for each technique
            for tech, color in palette.items():
                sp = sub[sub['technique'] == tech]
                if sp.empty:
                    continue
                grp = (sp
                       .groupby('IPC')[metric]
                       .agg(['mean', 'std'])
                       .reset_index())
                ax.plot(grp['IPC'], grp['mean'],
                        marker='o', label=tech, color=color)
                ax.fill_between(
                    grp['IPC'],
                    grp['mean'] - grp['std'],
                    grp['mean'] + grp['std'],
                    alpha=0.2, color=color
                )

            # Línea base para 'Full-data'
            baseline_df = df_all_init[
                (df_all_init['method'] == 'Full-data') &
                (df_all_init['model']  == model)
            ]
            if not baseline_df.empty:
                baseline_val = baseline_df[metric].mean()
                ax.axhline(
                    y=baseline_val,
                    linestyle='--',
                    color='gray',
                    label='Full-data'
                )

            ax.grid(True, linestyle='--', alpha=0.5)
            ax.set_xlim(0, max_ipc)

    # 6) Etiquetas globales de ejes
    # Para Matplotlib >= 3.4 puedes usar:
    fig.supxlabel(x_label, fontsize=14)
    fig.supylabel(y_label, fontsize=14, rotation='vertical', x=0)

    # 7) Leyenda común arriba
    # Take handles/labels from the last valid ax
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels,
               loc='upper center', ncol=len(palette)+1,
               title="Technique")
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

# Allows plotting evaluation in both reconstructed and latent spaces
def plot_metric_grid_all(
        df_full,
        df_only_coreset,
        df_latent,
        df_latent_ft,
        df_MLP=pd.DataFrame(),
        df_TDColer=pd.DataFrame(),
        spaces=('latent','original'),
        max_ipc=100,
        metric='test_balanced',
        methods=('k-means', 'k-center', 'ag'),
        models=None,
        x_label='Instances Per Class',
        y_label='Test balanced',
        name_savefig='figure.png'
    ):
    """
    Grid dinámico metric vs IPC:
      • Filas → cada modelo de `models`.
      • Columnas → bloques por espacio (cada bloque = len(methods) columnas).
    Fondo de cada bloque coloreado según espacio (original / latent).
    Dentro de cada celda:
      - Curvas media±std de cada técnica (Original, CorVAE, MLP-VAE, TDColer).
      - Línea base full-data.
      - Cuadro con valor Full-data y máximo de CorVAE.
    """

    # 1) Unir DataFrames con columna 'space' y 'technique'
    df_list = []
    for space in spaces:
        d_latent = df_latent[df_latent['space'] == space]
        d_mlp    = df_MLP[df_MLP['space'] == space] if not df_MLP.empty else pd.DataFrame() # <-- Agregado
        d_tdcol  = df_TDColer[df_TDColer['space'] == space] if not df_TDColer.empty else pd.DataFrame()
        
        for src, tech in [
            (df_only_coreset, 'Original'),
            (d_latent,       'CorVAE'),
            (d_mlp,          'MLP-VAE'), # <-- Agregado
            (d_tdcol,        'TDColer'),
        ]:
            if src.empty:
                continue
            tmp = src.copy()
            tmp['technique'] = tech
            tmp['space']     = space
            df_list.append(tmp)

    df_all = pd.concat(df_list, ignore_index=True)
    df_all = df_all[df_all['IPC'] <= max_ipc]

    # 2) Preparo figura y ejes
    n_rows = len(models)
    n_cols = len(spaces) * len(methods)
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(4*n_cols, 3*n_rows),
        sharex=True, sharey=True
    )
    axes = np.array(axes).reshape(n_rows, n_cols)

    # Fixed palette for lines (Se agregó MLP-VAE)
    tech_colors = {
        'Original': 'C0',
        'CorVAE':   'C2',
        'MLP-VAE':  'C3', # Color extra (rojo/naranja de matplotlib)
        'TDColer':  'C4',
    }
    baseline_color = 'gray'

    # Pastel palette for space backgrounds
    bg_palette = sns.color_palette("pastel", len(spaces))
    bg_colors  = dict(zip(spaces, bg_palette))

    remap1 = {
        'k-means':  'KM',
        'k-center': 'K-Centers',
        'ag':       'AG',
        'lc':      'LC',
    }

    remap2 = {
        'logreg': 'LR',
        'svc':    'SVC',
        'rf':     'RF',
        'xgb':    'XGBoost',
        'naive_bayes':   'NB',
        'mlp':   'MLP',
        'knn':  'KNN',
    }

    # 3) Rellenar cada subplot
    for i, model in enumerate(models):
        for k, space in enumerate(spaces):
            for j, method in enumerate(methods):
                col = k * len(methods) + j
                ax = axes[i, col]

                # Background shading according to space
                rgba = mcolors.to_rgba(bg_colors[space], alpha=0.1)
                ax.set_facecolor(rgba)

                sub = df_all[
                    (df_all['model']  == model) &
                    (df_all['method'] == method) &
                    (df_all['space']  == space)
                ]
                if sub.empty:
                    ax.set_visible(False)
                    continue

                ax.set_title(f"{remap2[model]} – {remap1[method]}", loc='left', pad=6)

                # Draw curves and calculate CorVAE maximum
                corvae_max_val = None
                corvae_max_ipc = None
                for tech, color in tech_colors.items():
                    sp = sub[sub['technique'] == tech]
                    if sp.empty:
                        continue
                    
                    grp = sp.groupby('IPC')[metric].agg(['mean','std']).reset_index()
                    ax.plot(grp['IPC'], grp['mean'],
                            marker='o', label=tech, color=color)
                    ax.fill_between(
                        grp['IPC'],
                        grp['mean'] - grp['std'],
                        grp['mean'] + grp['std'],
                        alpha=0.2, color=color
                    )
                    # if it is CorVAE, calculate its maximum
                    if tech == 'CorVAE':
                        idx = grp['mean'].idxmax()
                        corvae_max_val = grp.at[idx, 'mean']
                        corvae_max_ipc = grp.at[idx, 'IPC']

                # Full-data baseline
                baseline = df_full[df_full['model'] == model][metric].mean()
                ax.axhline(y=baseline,
                           linestyle='--',
                           color=baseline_color,
                           label='Full-data')

                ax.grid(True, linestyle='--', alpha=0.5)
                ax.set_xlim(0, max_ipc)

    fig.supxlabel(x_label, fontsize=14)
    fig.supylabel(y_label, fontsize=14, rotation='vertical', x=0.02)

    # Construcción dinámica de la leyenda basada en las técnicas presentes
    legend_elements = [
        Line2D([0], [0], marker='o', color=tech_colors['Original'], label='Original'),
        Line2D([0], [0], marker='o', color=tech_colors['CorVAE'],   label='CorVAE'),
    ]
    
    if 'MLP-VAE' in df_all['technique'].unique():
        legend_elements.append(
            Line2D([0], [0], marker='o', color=tech_colors['MLP-VAE'], label='MLP-VAE')
        )
        
    if 'TDColer' in df_all['technique'].unique():
        legend_elements.append(
            Line2D([0], [0], marker='o', color=tech_colors['TDColer'], label='TDColer')
        )
        
    legend_elements.append(
        Line2D([0], [0], linestyle='--', color=baseline_color,      label='Full-data')
    )

    fig.legend(
        handles=legend_elements,
        loc='upper center',
        ncol=len(legend_elements),
        bbox_to_anchor=(0.5, 1.03),  # sube un poco más la leyenda
        fontsize=12,
        title_fontsize=13
    )

    # Adjust layout to avoid overlapping
    plt.tight_layout(rect=[0, 0, 1, 0.97])  # more space at the top
    plt.savefig(name_savefig, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
import sys
# Insert at the beginning to give priority
sys.path.insert(0, 'utils')

from sklearn.metrics import silhouette_score
import tqdm
from distill import (
    distill_with_agglomerative,
    distill_with_kcenters,
    distill_least_confidence,
    distill_with_kmeans
    
)

def plot_silhouette_vs_ipc_all_methods(
    X_original: np.ndarray,
    y:           np.ndarray,
    ipc_values:  list[int] = list(range(10, 101, 10)),
    seeds:       list[int] = [5, 6, 7, 8, 9],
    model=None,           # solo para LC
    flat:        bool = True,
    figname = None,
    z_root_path: str ='results/checkpoint/checkpoint_shoppers_Dlatent_ft_F/vae',
):
    """
    Traza en una figura 2×2 la Silhouette vs IPC para:
      - AG  (Agglomerative)
      - K-Centers
      - LC  (Least Confidence)
      - KM  (KMeans-based distillation)
    """

    methods = ['AG', 'K-Centers', 'LC', 'KM']
    n_meth, n_seeds, n_ipc = len(methods), len(seeds), len(ipc_values)

    # Results: [método, seed, ipc]
    sil_orig = np.zeros((n_meth, n_seeds, n_ipc))
    sil_lat  = np.zeros_like(sil_orig)

    # --- Silhouette calculation per method / seed / IPC ---
    for m_idx, method in enumerate(methods):
        for i, seed in enumerate(seeds):
            # load and prepare latent
            Z = np.load(f'{z_root_path}/train_z_seed_{seed}.npy')
            if flat and Z.ndim == 3:
                n, t, d = Z.shape
                Z = Z.reshape(n, t * d)
            elif not flat and Z.ndim == 3:
                Z = Z[:, 0, :]
            # else, Z ya es 2D

            for j, ipc in enumerate(tqdm.tqdm(ipc_values, desc=f'{method} seed {seed}')):
                if method == 'AG':
                    Xo, yo = distill_with_agglomerative(  X_original, y, num_clusters=ipc, seed=seed)
                    Xl, yl = distill_with_agglomerative(       Z, y, num_clusters=ipc, seed=seed)

                elif method == 'K-Centers':
                    Xo, yo = distill_with_kcenters(  X_original, y, num_centroids=ipc, seed=seed)
                    Xl, yl = distill_with_kcenters(       Z, y, num_centroids=ipc, seed=seed)

                elif method == 'LC':
                    Xo, yo = distill_least_confidence(  X_original, y, num_samples=ipc, model=model, seed=seed)
                    Xl, yl = distill_least_confidence(       Z, y, num_samples=ipc, model=model, seed=seed)

                elif method == 'KM':
                    Xo, yo = distill_with_kmeans(  X_original, y, num_centroids=ipc, seed=seed)
                    Xl, yl = distill_with_kmeans(       Z, y, num_centroids=ipc, seed=seed)

                else:
                    raise ValueError(f"Método desconocido: {method!r}")

                # silhouette_score requires ≥2 clusters
                sil_orig[m_idx, i, j] = (
                    silhouette_score(Xo, yo)
                    if len(np.unique(yo)) > 1 else np.nan
                )
                sil_lat[m_idx, i, j] = (
                    silhouette_score(Xl, yl)
                    if len(np.unique(yl)) > 1 else np.nan
                )

    # --- Statistics over seeds ---
    mean_orig = np.nanmean(sil_orig, axis=1)  # [método, ipc]
    std_orig  = np.nanstd( sil_orig, axis=1)
    mean_lat  = np.nanmean(sil_lat,  axis=1)
    std_lat   = np.nanstd( sil_lat,  axis=1)

    # --- Draw the 4 subplots ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('Silhouette Score', fontsize=16)

    for m_idx, method in enumerate(methods):
        ax = axes.flat[m_idx]

        # Original
        ax.plot(ipc_values, mean_orig[m_idx], marker='o', label='Original')
        ax.fill_between(
            ipc_values,
            mean_orig[m_idx] - std_orig[m_idx],
            mean_orig[m_idx] + std_orig[m_idx],
            alpha=0.2
        )

        # Latent
        ax.plot(ipc_values, mean_lat[m_idx], marker='o', label='Latent')
        ax.fill_between(
            ipc_values,
            mean_lat[m_idx] - std_lat[m_idx],
            mean_lat[m_idx] + std_lat[m_idx],
            alpha=0.2
        )

        ax.set_title(method)
        if m_idx in (2, 3):
            ax.set_xlabel('IPC')
        if m_idx in (0, 2):
            ax.set_ylabel('Silhouette')

        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout(rect=[0, 0, 1, 0.95])  # leave space for the suptitle

    if figname:
        plt.savefig(figname)
    plt.show()

In [ ]:
from sklearn.metrics import pairwise_distances
from utils import reconstruct_data, load_reconstructed_data  
import json
from sklearn.manifold import TSNE

def plot_kmeans_distill_and_reconstruct_tsne_seaborn(
    X_original: np.ndarray,
    Z:            np.ndarray,
    y:            np.ndarray,
    ipc:          int,
    seed:         int,
    metadata_path: str,
    vae_dir:       str,
    device:        str,
    num_scaler,
    cat_encoder,
    label_encoder,
    concat_label:  bool,
    hyperparams_vae_path: str,
    figname:       str = None,
    method=None
):
    """
    - Distilla con KMeans(ipc) por clase en original y latente.
    - Reconstruye las muestras latentes seleccionadas al espacio original.
    - Aplica t-SNE a [X_original; Xo; X_rec].
    - Grafica con seaborn:
      • Puntos originales (suaves rojo/azul, alpha bajo).
      • Centroides originales (marker 'X', verde/green-yellow).
      • Reconstruidas (marker 'P', turquesa/violeta).
      • Leyenda separada para clases y para tipos de puntos.
    """

    if method == 'AG':
        Xo, yo = distill_with_agglomerative(  X_original, y, num_clusters=ipc, seed=seed)
        distill_z, distill_y = distill_with_agglomerative(Z, y, num_clusters=ipc, seed=seed)
    if method == 'KM':
        # 1) Distilación Original
        Xo, yo = distill_with_kmeans(
            X_original, y, num_centroids=ipc, seed=seed, get_closest=False
        )
        # 2) Distilación Latent
        distill_z, distill_y = distill_with_kmeans(
            Z, y, num_centroids=ipc, seed=seed, get_closest=False
        )
    elif method == 'K-Centers':
        Xo, yo = distill_with_kcenters(  X_original, y, num_centroids=ipc, seed=seed)
        distill_z, distill_y = distill_with_kcenters(       Z, y, num_centroids=ipc, seed=seed)

    elif method == 'LC':
        Xo, yo = distill_least_confidence(  X_original, y, num_samples=ipc, model=None, seed=seed)
        distill_z, distill_y = distill_least_confidence(       Z, y, num_samples=ipc, model=None, seed=seed)

    # 3) Cargar hiperparámetros VAE
    with open(hyperparams_vae_path, 'r', encoding='utf-8') as f:
        hyperparams_vae = json.load(f)['best_params']
    # 4) Reconstruir latente → original
    df_recon, _ = reconstruct_data(
        distill_z, distill_y,
        models_paths=vae_dir, device=device,
        json_config_path=metadata_path,
        latent_space=True, hyperparams=hyperparams_vae,
        method='k-means', ipc=ipc,
        concat_label=concat_label, seed=seed
    )
    X_rec, y_rec = load_reconstructed_data(
        df_recon, metadata_path,
        num_scaler=num_scaler,
        cat_encoder=cat_encoder,
        label_encoder=label_encoder
    )
    # 5) TSNE sobre combinado
    X_comb = np.vstack([X_original, Xo, X_rec])
    tsne = TSNE(n_components=2, random_state=seed)
    emb = tsne.fit_transform(X_comb)
    n = X_original.shape[0]
    m = Xo.shape[0]
    emb_orig = emb[:n]
    emb_o    = emb[n:n+m]
    emb_rec  = emb[n+m:]
    # 6) Paletas
    palette_cls = {0: 'deepskyblue', 1: 'darkgreen'}
    palette_o   = {0: 'darkblue', 1: 'purple'}
    palette_r   = {0: 'hotpink', 1: 'darkorange'}
    # 7) Plot
    sns.set_style('white')
    fig, ax = plt.subplots(figsize=(8, 6))
    # Original: smoothed
    sns.scatterplot(
        x=emb_orig[:,0], y=emb_orig[:,1],
        hue=y, palette=palette_cls,
        legend='brief', alpha=0.3, s=15, ax=ax
    )
    # Capture class legend
    handles_cls, labels_cls = ax.get_legend_handles_labels()

        # Reconstructed
    for cls in np.unique(y_rec):
        idx_cls = np.where(y_rec == cls)[0]
        ax.scatter(
            emb_rec[idx_cls,0], emb_rec[idx_cls,1],
            marker='o', s=80,
            edgecolors=palette_r[cls],          # red border
    linewidths=3, 
            label=f'Recons {cls}'
        )
    
    # Original centroids
    for cls in np.unique(yo):
        idx_cls = np.where(yo == cls)[0]
        ax.scatter(
            emb_o[idx_cls,0], emb_o[idx_cls,1],
            marker='X',
            s=80,
            #c=palette_o[cls],
            linewidths=0.5,              # thinner line width
            edgecolors=palette_o[cls],   # make sure the border is drawn
            label=f'Orig {cls}'
        )

    # Separate legends
    leg1 = ax.legend(handles=handles_cls, labels=labels_cls,
                     title='Class', loc='upper left')
    # Get only centroids and reconstructed handles
    handles2, labels2 = [], []
    for handle, label in zip(*ax.get_legend_handles_labels()):
        if 'Orig' in label or 'Recon' in label:
            handles2.append(handle)
            labels2.append(label)
    leg2 = ax.legend(handles=handles2, labels=labels2,
                     title='Point type', loc='upper right')
    ax.add_artist(leg1)
    ax.set_title(f't-SNE - {method} (IPC={ipc})')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.grid(False)
    plt.tight_layout()
    if figname:
        plt.savefig(figname)
        print(f'Figura guardada en: {figname}')
    plt.show()

In [ ]:
def load_latent(seed, flat, z_root_path, cls=False):
    Z = np.load(f'{z_root_path}/train_z_seed_{seed}.npy')
    if flat and Z.ndim == 3:
        n, t, d = Z.shape
        return Z.reshape(n, t * d)
    elif cls:
        return Z[:, 0, :]
    else:
        return Z



In [ ]:
# Funciones para renombrar y adecuar los nombres de los 
# resultados de TDCoLER

def rename_rows(df, column, mapping):
    """
    Renombra las filas de un DataFrame para que coincidan con el método.
    """
    df = df.copy()

    # Aplicamos el cambio en el DataFrame
    df[column] = (df[column]
                .replace(mapping)                # reemplaza los que estén en el dict
                #.str.lower()                     # opcional: pasar todo a minúsculas
                )
    return df

def preprocess_baseline(df):
    df = df[['Classifier','Encoder','Data Parse Mode','Distill Method','Distill Space',
             'Output Space','Convert Binary','Cluster Center','N','Score']]
    
    mapping_classifier = {
        'XGBClassifier':       'xgb',
        'MLPClassifier':  'mlp',
        'KNeighborsClassifier':'knn',
        'GaussianNB': 'naive_bayes',
        'LogisticRegression': 'logreg',
    }

    mapping_method = {
        'KMeans':       'k-means',
        'Agglo':  'ag',
    }

    mapping_space = {
        'decoded':       'original',
        'encoded':  'latent',

    }

    df = rename_rows(df, 'Classifier', mapping_classifier)
    df = rename_rows(df, 'Distill Method', mapping_method)
    df = rename_rows(df, 'Output Space', mapping_space)

    df.rename(columns={'N': 'IPC',
                       'Distill Method':'method',
                       'Classifier':'model',
                       'Score':'test_balanced',
                       'Output Space':'space'
                       
                       }, inplace=True)

    df_full = df[
                        (df['Data Parse Mode'] == 'mixed') &
                        (df['method'] == 'Original') &
                        (df['Convert Binary'] == False)
                    ]
    
    df_only_coreset = df[
                        (df['method'].isin(['k-means', 'ag'])) &
                        (df['Distill Space'] == 'original') &
                        (df['space'] == 'original') &
                        (df['Convert Binary'] == False) &
                        (df['Cluster Center'] == 'centroid') 
                    ]
                    
    df_latent = df[
                        (df['method'].isin(['k-means', 'ag'])) &
                        (df['Distill Space'] == 'encoded') &
                        (df['Convert Binary'] == False) &
                        (df['Encoder'] == 'TF')
                    ]
    df_latent_ft = df[
                        (df['method'].isin(['k-means', 'ag'])) &
                        (df['Distill Space'] == 'encoded') &
                        (df['Convert Binary'] == False) &
                        (df['Encoder'] == 'TF-MultiHead')
                    ]

    df_full = df_full[['model','method','IPC','test_balanced','space']]
    df_only_coreset = df_only_coreset[['model','method','IPC','test_balanced','space']]
    df_latent = df_latent[['model','method','IPC','test_balanced','space']]
    df_latent_ft = df_latent_ft[['model','method','IPC','test_balanced', 'space']]

    return df_full,df_only_coreset,df_latent, df_latent_ft

In [ ]:
def summarize_two_metrics(
    ipc: float,
    df_only_coreset: pd.DataFrame,
    df_latent: pd.DataFrame,
    df_tdcoler: pd.DataFrame,
    df_latent_ft: pd.DataFrame = None,
    df_mlp: pd.DataFrame = None,          # ← nuevo
) -> pd.DataFrame:

    df_list = []

    tmp = df_only_coreset.copy()
    tmp['technique'] = 'Only-Coreset'
    tmp['space']     = 'original'
    df_list.append(tmp)

    tmp = df_latent.copy()
    tmp['technique'] = 'CorVAE-recon'
    df_list.append(tmp)



    if df_latent_ft is not None and not df_latent_ft.empty:
        tmp = df_latent_ft.copy()
        tmp['technique'] = 'CorVAE-latent'
        df_list.append(tmp)

    if not df_tdcoler.empty:
        tmp = df_tdcoler.copy()
        tmp['technique'] = 'TDColer'
        df_list.append(tmp)

    if df_mlp is not None and not df_mlp.empty:   # ← nuevo bloque
        tmp = df_mlp.copy()
        tmp['technique'] = 'MLP-VAE'
        df_list.append(tmp)

    df_all = pd.concat(df_list, ignore_index=True)
    df_all['method'] = df_all['method'].replace({'k_center': 'k-center'}) 
    df_all = df_all[~df_all['method'].isin(['craig', 'random'])]
    df_all = df_all[df_all['model'] != 'knn']
    df_ipc = df_all[df_all['IPC'] == ipc]

    def pivot_metric(df: pd.DataFrame, metric: str) -> pd.DataFrame:
        stats = (
            df
            .groupby(['technique','space','model','method'])[metric]
            .agg(mean='mean', std='std')
            .reset_index()
        )
        stats['fmt'] = stats.apply(
            lambda r: f"{r['mean']:.3f}±{r['std']:.3f}", axis=1
        )
        table = (
            stats
            .pivot_table(
                index=['technique','space','model'],
                columns='method',
                values='fmt',
                aggfunc='first'
            )
        )
        return table.rename(columns={
            'k-means':  'KM',
            'ag':       'AG',
            'k-center': 'K-center',
            'lc':       'LC'
        })

    table_bal = pivot_metric(df_ipc, 'test_balanced')
    table_roc = pivot_metric(df_ipc, 'test_roc_auc')

    combined = pd.concat(
        [table_bal, table_roc],
        axis=1,
        keys=['test_balanced', 'test_roc_auc']
    )
    combined = combined.reset_index()

    combined['technique'] = combined['technique'].replace({
        'Only-Coreset':  'Only-Coreset',
        'CorVAE-recon':  'CorVAE',
        'CorVAE-latent': 'CorVAE',
        'TDColer':       'TDColer',
        'MLP-VAE':       'MLP-VAE',
    })
    combined['model'] = combined['model'].replace({
        'logreg':      'Logistic Regression',
        'mlp':         'MLP',
        'naive_bayes': 'Naive Bayes',
        'xgb':         'XGBoost',
        'rf':          'Random Forest',
        'svc':         'SVC',
        'svm':         'SVM',
    })
    combined['space'] = combined['space'].replace({
        'latent':   'Latent',
        'original': 'Reconstructed',
    })
    
    return co-mbined

In [ ]:
# Carga de MLP-VAE baselines para todos los datasets
df_mlp_adult    = pd.read_csv('checkpoint_adult_MLP_Dlatent/metrics.csv')
df_mlp_default  = pd.read_csv('checkpoint_default_MLP_Dlatent/metrics.csv')
df_mlp_shoppers = pd.read_csv('checkpoint_shoppers_MLP_Dlatent/metrics.csv')

## Preprocessing of TDCOLER results

In [ ]:
df_tdcoler_adult = pd.read_csv('TDCOLER_data/TDCOLER_adult.csv')
df_tdcoler_default = pd.read_csv('TDCOLER_data/TDCOLER_default.csv')

(df_tdcoler_full_adult, df_tdcoler_only_coreset_adult, 
 df_tdcoler_latent_adult, df_tdcoler_latent_ft_adult) = preprocess_baseline(df_tdcoler_adult)

(df_tdcoler_full_default, df_tdcoler_only_coreset_default, 
 df_tdcoler_latent_default, df_tdcoler_latent_ft_default) = preprocess_baseline(df_tdcoler_default)

## Paysim Dataset

In [ ]:
tabla = summarize_two_metrics(
    ipc=10,
    df_only_coreset=df_only_coreset_paysim,
    df_latent=df_latent_paysim,
    df_tdcoler=pd.DataFrame(),
    df_latent_ft=None,
    df_mlp=df_mlp_paysim
)
tabla

In [ ]:
# ── CARGA DATOS PAYSIM ───────────────────────────────────────────────────────
df_paysim_corvae = pd.read_csv('checkpoint_paysim_Dlatent_I/metrics.csv')
df_only_coreset_paysim = pd.read_csv('checkpoint_paysim_Doriginal_I/metrics.csv')

df_full_paysim         = df_paysim_corvae[df_paysim_corvae['method'] == 'Full-data']
df_only_coreset_paysim = df_only_coreset_paysim[df_paysim_corvae['space'] == 'original']
df_latent_paysim       = df_paysim_corvae[df_paysim_corvae['method'] != 'Full-data']
df_latent_ft_paysim    = pd.DataFrame()   # sin versión fine-tuned

df_mlp_paysim = pd.read_csv('checkpoint_paysim_MLP_Dlatent/metrics.csv')

# Paysim – balanced accuracy, KM + AG
methods = ['k-means', 'ag']
models  = ['logreg', 'mlp', 'naive_bayes', 'xgb', 'rf']

plot_metric_grid_all(
    df_full=df_full_paysim,
    df_only_coreset=df_only_coreset_paysim,
    df_latent=df_latent_paysim,
    df_latent_ft=df_latent_ft_paysim,
    df_MLP=df_mlp_paysim,
    metric='test_balanced',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='balanced accuracy',
    name_savefig='results/curve_figures/paysim_MLP_KM_AG_balanced_accuracy.png',
    spaces=('latent','original')
)

# # Paysim – balanced accuracy, KM + AG, modelos RF y KNN
# methods = ['k-means', 'ag']
# models  = ['rf', 'knn']

# plot_metric_grid_all(
#     df_full=df_full_paysim,
#     df_only_coreset=df_only_coreset_paysim,
#     df_latent=df_latent_paysim,
#     df_latent_ft=df_latent_ft_paysim,
#     df_MLP=df_mlp_paysim,
#     metric='test_balanced',
#     methods=methods,
#     models=models,
#     x_label='IPC',
#     y_label='balanced accuracy',
#     name_savefig='results/curve_figures/paysim_rf_knn_KM_AG_balanced_accuracy.png'
# )

# # Paysim – ROC AUC, KM + AG
# methods = ['k-means', 'ag']
# models  = ['logreg', 'mlp', 'naive_bayes', 'xgb', 'rf', 'knn']

# plot_metric_grid_all(
#     df_full=df_full_paysim,
#     df_only_coreset=df_only_coreset_paysim,
#     df_latent=df_latent_paysim,
#     df_latent_ft=df_latent_ft_paysim,
#     df_MLP=df_mlp_paysim,
#     metric='test_roc_auc',
#     methods=methods,
#     models=models,
#     x_label='IPC',
#     y_label='ROC AUC',
#     name_savefig='results/curve_figures/paysim_KM_AG_roc_auc.png'
#

## Adult Dataset

In [ ]:
df_only_coreset_adult = pd.read_csv('results/checkpoint/checkpoint_adult_Doriginal_G/metrics.csv')

df_full_adult = df_only_coreset_adult[df_only_coreset_adult['method'] == 'Full-data']
df_latent_adult = pd.read_csv('results/checkpoint/checkpoint_adult_Dlatent_G/metrics.csv')
df_latent_ft_adult = pd.read_csv('results/checkpoint/checkpoint_adult_Dlatent_ft_G/metrics.csv')

In [ ]:
df_only_coreset_adult.head()

In [ ]:
df_full_adult.columns

In [ ]:
# Informacion de la evaluación con todos los datos
df_full_adult[['model','method','optuna_time','train_time']]

### Table calculation with IPC values

In [ ]:
tabla = summarize_two_metrics(
    ipc=10,
    df_only_coreset=df_only_coreset_adult,
    df_latent=df_latent_adult,
    df_tdcoler=df_tdcoler_latent_adult,
    df_latent_ft=None,
    df_mlp=df_mlp_adult
)
tabla

Adult

In [ ]:
tabla = summarize_two_metrics(
    ipc=50,
    df_only_coreset=df_only_coreset_adult,
    df_latent=df_latent_adult,
    df_tdcoler=df_tdcoler_latent_adult,
    df_latent_ft=None,
    df_mlp=df_mlp_adult
)
tabla

In [ ]:
tabla = summarize_two_metrics(
    ipc=100,
    df_only_coreset=df_only_coreset_adult,
    df_latent=df_latent_adult,
    df_tdcoler=df_tdcoler_latent_adult,
    df_latent_ft=None,
    df_mlp=df_mlp_adult
)
tabla

### Curve calculation for an IPC range from 10 to 100

In [ ]:
# Curvas de evaluación en reconstruido y latente con la metricas de balanced accuracy
# para K-means y agglomerative clustering
methods = ['k-means','ag']
models = ['logreg','mlp','naive_bayes','xgb']

compare_columns = df_tdcoler_latent_adult.columns

plot_metric_grid_all(
    df_full=df_full_adult[compare_columns],
    df_only_coreset=df_only_coreset_adult[compare_columns],
    df_latent=df_latent_adult[compare_columns],
    df_latent_ft=df_latent_ft_adult[compare_columns],
    df_TDColer=df_tdcoler_latent_adult,    
    df_MLP=df_mlp_adult,  
    metric='test_balanced',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='balanced accuracy',
    name_savefig='results/curve_figures/adult_tdcoler_KM_AG_balanced_accuracy.png'
)

In [ ]:
# Curvas de evaluación en reconstruido y latente con la metricas de balanced accuracy
# para K-means y agglomerative clustering pero para los otros modelos de SVC y RF
methods = ['k-means','ag']
models = ['rf','svc']

compare_columns = df_tdcoler_latent_adult.columns

plot_metric_grid_all(
    df_full=df_full_adult[compare_columns],
    df_only_coreset=df_only_coreset_adult[compare_columns],
    df_latent=df_latent_adult[compare_columns],
    df_latent_ft=df_latent_ft_adult[compare_columns],
    df_TDColer=df_tdcoler_latent_adult,    
    df_MLP=df_mlp_adult, 
    metric='test_balanced',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='balanced accuracy',
    name_savefig='results/curve_figures/adult_svc_rf_KM_AG_balanced_accuracy.png'
)

In [ ]:
# Curvas de evaluación en reconstruido y latente con la metricas de balanced accuracy
# para least confidence y k-centers para todos los modelos
methods = ['lc','k-center']
models = ['knn','logreg','mlp','naive_bayes','xgb','rf','svc']

compare_columns = df_tdcoler_latent_adult.columns

plot_metric_grid_all(
    df_full=df_tdcoler_full_adult,
    df_only_coreset=df_only_coreset_adult[compare_columns],
    df_latent=df_latent_adult[compare_columns],
    df_latent_ft=df_latent_ft_adult[compare_columns],
    df_TDColer=df_tdcoler_latent_adult,
    df_MLP=df_mlp_adult, 
    #space='original',      
    metric='test_balanced',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='balanced accuracy',
    name_savefig='results/curve_figures/adult_LC_KCENTERS_balanced_accuracy.png'
)

In [ ]:
# Curvas de evaluación en reconstruido y latente con la metricas de ROC AUC
# para kmeans y agglomerative clustering

methods = ['k-means','ag']
models = ['logreg','mlp','naive_bayes','xgb','rf','svc']

compare_columns = df_full_adult.columns

plot_metric_grid_all(
    df_full=df_full_adult[compare_columns],
    df_only_coreset=df_only_coreset_adult[compare_columns],
    df_latent=df_latent_adult[compare_columns],
    df_latent_ft=df_latent_ft_adult[compare_columns],
    df_MLP=df_mlp_adult, 
    metric='test_roc_auc',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='ROC AUC',
    name_savefig='results/curve_figures/adult_KM_AG_roc_auc.png'
)

In [ ]:
# Curvas de evaluación en reconstruido y latente con la metricas de ROC AUC
# para LC y K-centers para todos los modelos

methods = ['lc','k-center']
models = ['knn','logreg','mlp','xgb']
compare_columns = df_full_adult.columns

plot_metric_grid_all(
    df_full=df_full_adult[compare_columns],
    df_only_coreset=df_only_coreset_adult[compare_columns],
    df_latent=df_latent_adult[compare_columns],
    df_latent_ft=df_latent_ft_adult[compare_columns],
    metric='test_roc_auc',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='ROC AUC',
    name_savefig='results/curve_figures/adult_LC_KCENTERS_roc_auc.png'
)

## Default Dataset

In [ ]:
df_only_coreset_default = pd.read_csv('results/checkpoint/checkpoint_default_Doriginal_H/metrics.csv')

df_full_default = df_only_coreset_default[df_only_coreset_default['method'] == 'Full-data']
df_latent_default = pd.read_csv('results/checkpoint/checkpoint_default_Dlatent_G/metrics.csv')
df_latent_ft_default = pd.read_csv('results/checkpoint/checkpoint_default_Dlatent_ft_H/metrics.csv')

In [ ]:
df_only_coreset_default.head()

In [ ]:
# Información de la evaluación con todos los datos
df_full_default[['model','method','optuna_time']]

In [ ]:
df_only_coreset_default

### Table calculation with IPC values

In [ ]:
tabla = summarize_two_metrics(
    ipc=10,
    #metric='test_balanced',
    df_only_coreset=df_only_coreset_default,
    df_latent=df_latent_default,
    df_tdcoler=df_tdcoler_latent_default,
    df_latent_ft=None,
    df_mlp=df_mlp_default
)
tabla

In [ ]:
tabla = summarize_two_metrics(
    ipc=50,
    #metric='test_balanced',
    df_only_coreset=df_only_coreset_default,
    df_latent=df_latent_default,
    df_tdcoler=df_tdcoler_latent_default,
    df_latent_ft=None
)
tabla

In [ ]:
tabla = summarize_two_metrics(
    ipc=100,
    #metric='test_balanced',
    df_only_coreset=df_only_coreset_default,
    df_latent=df_latent_default,
    df_tdcoler=df_tdcoler_latent_default,
    df_latent_ft=None
)
tabla

### Curve calculation for an IPC range from 10 to 100

In [ ]:
methods = ['k-means','ag']
models = ['logreg','mlp','naive_bayes','xgb']

compare_columns = df_tdcoler_latent_default.columns

plot_metric_grid_all(
    df_full=df_full_default[compare_columns],
    df_only_coreset=df_only_coreset_default[compare_columns],
    df_latent=df_latent_default[compare_columns],
    df_latent_ft=df_latent_ft_default[compare_columns],
    df_TDColer=df_tdcoler_latent_default,
    df_MLP=df_mlp_default, 
    metric='test_balanced',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='balanced accuracy',
    name_savefig='results/curve_figures/default_tdcoler_KM_AG_balanced_accuracy.png'
)

In [ ]:
methods = ['k-means','ag']
models = ['rf','svc']
compare_columns = df_tdcoler_latent_default.columns

plot_metric_grid_all(
    df_full=df_full_default[compare_columns],
    df_only_coreset=df_only_coreset_default[compare_columns],
    df_latent=df_latent_default[compare_columns],
    df_latent_ft=df_latent_ft_default[compare_columns],
    df_TDColer=df_tdcoler_latent_default,
    df_MLP=df_mlp_default, 
    metric='test_balanced',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='balanced accuracy',
    name_savefig='results/curve_figures/default_svc_rf_KM_AG_balanced_accuracy.png'
)

In [ ]:
methods = ['lc','k-center']
models = ['logreg','mlp','naive_bayes','xgb','rf','svc']

compare_columns = df_tdcoler_latent_default.columns

plot_metric_grid_all(
    df_full=df_tdcoler_full_default,
    df_only_coreset=df_only_coreset_default[compare_columns],
    df_latent=df_latent_default[compare_columns],
    df_latent_ft=df_latent_ft_default[compare_columns],
    df_TDColer=df_tdcoler_latent_default,
    #space='original',      
    metric='test_balanced',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='balanced accuracy',
    name_savefig='results/curve_figures/default_LC_KCENTERS_balanced_accuracy.png'
)

In [ ]:
methods = ['k-means','ag']
models = ['logreg','mlp','naive_bayes','xgb','rf','svc']

compare_columns = df_full_default.columns

plot_metric_grid_all(
    df_full=df_full_default[compare_columns],
    df_only_coreset=df_only_coreset_default[compare_columns],
    df_latent=df_latent_default[compare_columns],
    df_latent_ft=df_latent_ft_default[compare_columns],    
    metric='test_roc_auc',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='ROC AUC',
    name_savefig='results/curve_figures/default_KM_AG_roc_auc.png'
)

In [ ]:
methods = ['lc','k-center']
models = ['knn','logreg','mlp','xgb','rf','svc']
compare_columns = df_full_default.columns

plot_metric_grid_all(
    df_full=df_full_default[compare_columns],
    df_only_coreset=df_only_coreset_default[compare_columns],
    df_latent=df_latent_default[compare_columns],
    df_latent_ft=df_latent_ft_default[compare_columns],
    metric='test_roc_auc',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='ROC AUC',
    name_savefig='results/curve_figures/default_LC_KCENTERS_roc_auc.png'
)

### Efficiency metrics calculation for default

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from matplotlib.patches import Patch, Rectangle
from matplotlib.legend_handler import HandlerBase

class _Blank(HandlerBase):
    def create_artists(self, legend, orig_handle,
                       xdescent, ydescent, width, height, fontsize, trans):
        return [Rectangle((0,0), 0, 0, alpha=0, transform=trans)]

rename_map = {'rf':'RF','xgb':'XGBoost','mlp':'MLP',
              'naive_bayes':'NB','logreg':'LR'}

def filter_and_agg(df, method='k-means', ipc=10):
    return (
        df.loc[(df['method']==method) &
               (df['IPC']==ipc) &
               (~df['model'].isin(['knn', 'svc']))]
        .groupby('model', as_index=False)
        .agg({'encoder_inference_time':'mean','decoder_inference_time':'mean',
              'optuna_time':'mean','destillation_time':'mean'})
        .assign(model=lambda d: d['model'].replace(rename_map))
    )

def get_components(df, models):
    d = df.set_index('model').loc[models]
    return ((d['encoder_inference_time']+d['decoder_inference_time']).values,
             d['optuna_time'].values, d['destillation_time'].values)

# ── Datos ─────────────────────────────────────────────────────
df_full_t = (df_full_paysim
    .loc[~df_full_paysim['model'].isin(['knn', 'svc']), ['model','optuna_time']].copy())
df_full_t['model'] = df_full_t['model'].replace(rename_map)
df_full_t = df_full_t.groupby('model', as_index=False)['optuna_time'].mean()

df_lat_p = filter_and_agg(df_latent_paysim)
df_mlp_p = filter_and_agg(df_mlp_paysim)
models   = df_lat_p['model'].tolist()
opt_full = df_full_t.set_index('model').loc[models, 'optuna_time'].values

vae_lat, opt_lat, dist_lat = get_components(df_lat_p, models)
vae_mlp, opt_mlp, dist_mlp = get_components(df_mlp_p, models)
total_lat = vae_lat + opt_lat + dist_lat
total_mlp = vae_mlp + opt_mlp + dist_mlp

C = dict(vae_cor='#2ca02c', opt_cor='#ff9896', dis_cor='#d62728',
         vae_mlp='#1f77b4', opt_mlp='#aec7e8', dis_mlp='#9467bd', full='#17becf')
EPS   = 0.01
idx   = np.arange(len(models))
width = 0.25

def draw_stacked(ax, xpos, vae, opt, dist, cv, co, cd, base=0):
    b = np.full(len(models), base)
    ax.bar(xpos, vae,  width, bottom=b,            color=cv)
    ax.bar(xpos, opt,  width, bottom=b+vae,         color=co)
    ax.bar(xpos, dist, width, bottom=b+vae+opt,     color=cd)

# ── Figura ────────────────────────────────────────────────────
sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(12, 6))

draw_stacked(ax, idx-width, vae_lat, opt_lat, dist_lat,
             C['vae_cor'], C['opt_cor'], C['dis_cor'], base=EPS)
draw_stacked(ax, idx,       vae_mlp, opt_mlp, dist_mlp,
             C['vae_mlp'], C['opt_mlp'], C['dis_mlp'], base=EPS)
ax.bar(idx+width, opt_full, width,
       bottom=np.full(len(models), EPS), color=C['full'])

for j in range(len(models)):
    for x, t in [(idx[j]-width, total_lat[j]),
                 (idx[j],       total_mlp[j]),
                 (idx[j]+width, opt_full[j])]:
        ax.text(x, (EPS+t)*1.1, f"{t:.1f}", ha='center', va='bottom', fontsize=8)

ax.set_yscale('log')
ax.set_ylim(EPS*0.5, max(total_lat.max(), total_mlp.max(), opt_full.max())*3)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(
    lambda x,_: f"{x:.2f}" if x<1 else f"{x:,.0f}"))
ax.set_xticks(idx)
ax.set_xticklabels(models, fontsize=11)
ax.set_xlabel("Model", fontsize=12)
ax.set_ylabel("Time (s) — log scale", fontsize=12)
ax.set_title("Inference time comparison – PaySim\n"
             "(CorVAE vs MLP-VAE vs Full-data, k-means / IPC=10)", fontsize=12)

# ── Leyenda 3 columnas: col0=CorVAE | col1=MLP-VAE | col2=Full
sp1 = Patch(color='none', label='')
sp2 = Patch(color='none', label='')
h = [
    Patch(color=C['vae_cor'], label='VAE inference (CorVAE)'),
    Patch(color=C['vae_mlp'], label='VAE inference (MLP-VAE)'),
    Patch(color=C['full'],    label='Optuna full-data'),
    Patch(color=C['opt_cor'], label='Optuna coreset (CorVAE)'),
    Patch(color=C['opt_mlp'], label='Optuna coreset (MLP-VAE)'),
    sp1,
    Patch(color=C['dis_cor'], label='Distillation (CorVAE)'),
    Patch(color=C['dis_mlp'], label='Distillation (MLP-VAE)'),
    sp2,
]
ax.legend(handles=h, handler_map={sp1:_Blank(), sp2:_Blank()},
          ncol=3, loc='upper left', framealpha=0.92,
          fontsize=9, columnspacing=1.2, handlelength=1.2)

plt.tight_layout()
plt.savefig('results/efficiency_figures/paysim_tiempos_log.png',
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from matplotlib.patches import Patch, Rectangle
from matplotlib.legend_handler import HandlerBase

class _Blank(HandlerBase):
    def create_artists(self, legend, orig_handle,
                       xdescent, ydescent, width, height, fontsize, trans):
        return [Rectangle((0,0), 0, 0, alpha=0, transform=trans)]

# ── 1) Datos ──────────────────────────────────────────────────
lat_rf = (
    df_latent_paysim
    .loc[(df_latent_paysim['model']  == 'rf') &
         (df_latent_paysim['method'] == 'k-means') &
         (df_latent_paysim['IPC']    == 10)]
    .agg({'vae_pretrain_time':'mean','encoder_inference_time':'mean',
          'decoder_inference_time':'mean','destillation_time':'mean',
          'optuna_time':'mean'})
)

mlp_rf = (
    df_mlp_paysim
    .loc[(df_mlp_paysim['model']  == 'rf') &
         (df_mlp_paysim['method'] == 'k-means') &
         (df_mlp_paysim['IPC']    == 10)]
    .agg({'vae_pretrain_time':'mean','encoder_inference_time':'mean',
          'decoder_inference_time':'mean','destillation_time':'mean',
          'optuna_time':'mean'})
)

full_rf = df_full_paysim.loc[df_full_paysim['model']=='rf','optuna_time'].mean()

# ── 2) Componentes por barra ──────────────────────────────────
EPS = 0.001   # base log

bars = {
    'CorVAE': {
        'VAE inference':  lat_rf['encoder_inference_time'] + lat_rf['decoder_inference_time'],  # 1º
        'Optuna coreset': lat_rf['optuna_time'],                                                  # 2º
        'Distillation':   lat_rf['destillation_time'],                                           # 3º
        'VAE training':   lat_rf['vae_pretrain_time'],                                           # 4º arriba
    },
    'MLP-VAE': {
        'VAE inference':  mlp_rf['encoder_inference_time'] + mlp_rf['decoder_inference_time'],
        'Optuna coreset': mlp_rf['optuna_time'],
        'Distillation':   mlp_rf['destillation_time'],
        'VAE training':   mlp_rf['vae_pretrain_time'],
    },
    'Full-data': {
        'Optuna full-data': full_rf,
    },
}

C = {
    'VAE training':     '#1f77b4',
    'VAE inference':    '#2ca02c',
    'Distillation':     '#ff7f0e',
    'Optuna coreset':   '#ff9896',
    'Optuna full-data': '#17becf',
}

# ── 3) Tabla resumen ──────────────────────────────────────────
rows = []
for bar_name, comps in bars.items():
    row = {'Bar': bar_name}
    row.update(comps)
    rows.append(row)
print(pd.DataFrame(rows).fillna(0).to_string(index=False))

# ── 4) Plot ───────────────────────────────────────────────────
sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(9, 6))

idx   = np.arange(len(bars))
width = 0.5
bar_names = list(bars.keys())

for i, (bar_name, comps) in enumerate(bars.items()):
    bottom = EPS
    for comp_name, val in comps.items():
        ax.bar(i, val, width, bottom=bottom, color=C[comp_name],
               label=comp_name, zorder=3)
        # Etiqueta dentro del segmento si es suficientemente alto
        seg_top = bottom + val
        seg_mid = bottom + val / 2
        if val / (bottom + val) > 0.08:   # solo si el segmento ocupa >8% visual
            ax.text(i, seg_mid, f"{val:.2f}s",
                    ha='center', va='center', fontsize=7.5,
                    color='white', fontweight='bold', zorder=4)
        bottom = seg_top

    # Etiqueta de total encima de la barra
    total = sum(comps.values())
    ax.text(i, (EPS + total) * 1.12, f"{total:.1f} s",
            ha='center', va='bottom', fontsize=9.5,
            fontweight='bold', color='black', zorder=4)

# ── 5) Escala log ─────────────────────────────────────────────
all_totals = [sum(c.values()) for c in bars.values()]
ax.set_yscale('log')
ax.set_ylim(EPS * 0.3, max(all_totals) * 4)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(
    lambda x, _: f"{x:.3f}" if x < 0.01
    else (f"{x:.2f}" if x < 1 else f"{x:,.0f}")
))
# # Grid logarítmico: líneas mayores y menores
# ax.grid(True, which='major', linestyle='-',  alpha=0.4, zorder=0)
# ax.grid(True, which='minor', linestyle='--', alpha=0.2, zorder=0)
# ax.minorticks_on()

# ── 6) Ejes ───────────────────────────────────────────────────
ax.set_xticks(idx)
ax.set_xticklabels(bar_names, fontsize=12)
ax.set_xlabel("Pipeline", fontsize=12)
ax.set_ylabel("Time (s) — log scale", fontsize=12)
ax.set_title("Training time comparison – PaySim / RF\n"
             "(k-means, IPC = 10)", fontsize=12)

# ── 7) Leyenda sin duplicados ─────────────────────────────────
# Orden fijo: CorVAE col | MLP-VAE col | Full-data col
sp1 = Patch(color='none', label='')
sp2 = Patch(color='none', label='')
sp3 = Patch(color='none', label='')

h = [
    Patch(color=C['VAE inference'],    label='VAE inference (CorVAE)'),
    Patch(color=C['VAE inference'],    label='VAE inference (MLP-VAE)'),
    Patch(color=C['Optuna full-data'], label='Optuna full-data'),

    Patch(color=C['Optuna coreset'],   label='Optuna coreset (CorVAE)'),
    Patch(color=C['Optuna coreset'],   label='Optuna coreset (MLP-VAE)'),
    sp1,

    Patch(color=C['Distillation'],     label='Distillation (CorVAE)'),
    Patch(color=C['Distillation'],     label='Distillation (MLP-VAE)'),
    sp2,

    Patch(color=C['VAE training'],     label='VAE training (CorVAE)'),
    Patch(color=C['VAE training'],     label='VAE training (MLP-VAE)'),
    sp3,
]

ax.legend(handles=h,
          handler_map={sp1:_Blank(), sp2:_Blank(), sp3:_Blank()},
          ncol=3, loc='upper left', framealpha=0.92,
          fontsize=8.5, columnspacing=1.0, handlelength=1.2)

plt.tight_layout()
plt.savefig('results/efficiency_figures/paysim_rf_training_log.png',
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

paths = {
    'Adult': {
        'Original':  'data/adult/dataset.csv',
        'Distilled': 'results/checkpoint/checkpoint_adult_Dlatent_G/vae/reconstructed_data/IPC_10/method_k-means_seed_5.csv'
    },
    'Default': {
        'Original':  'data/default/dataset.csv',
        'Distilled': 'results/checkpoint/checkpoint_default_Dlatent_G/vae/reconstructed_data/IPC_10/method_k-means_seed_5.csv'
    },
    'PaySim': {
        'Original':  'data/paysim/paysim_dataset.csv',
        'Distilled': 'checkpoint_paysim_Dlatent_I/vae/reconstructed_data/IPC_10/method_k-means_seed_5.csv'
    },
    'Shoppers': {
        'Original':  'data/shoppers/dataset.csv',
        'Distilled': 'results/checkpoint/checkpoint_shoppers_Dlatent_F/vae/reconstructed_data/IPC_10/method_k-means_seed_5.csv'
    },
}

def resolve_csv(path):
    if os.path.exists(path):
        return path
    parent = os.path.dirname(path)
    candidates = glob.glob(os.path.join(parent, '*.csv'))
    if candidates:
        return candidates[0]
    raise FileNotFoundError(f"No CSV found in: {parent}")

# ── 1) Tamaños ────────────────────────────────────────────────
records = []
for ds, variants in paths.items():
    for tipo, fp in variants.items():
        real_fp = resolve_csv(fp)
        size_kb = os.path.getsize(real_fp) / 1024
        n_rows  = sum(1 for _ in open(real_fp, encoding='utf-8')) - 1
        records.append({'Dataset': ds, 'Type': tipo,
                        'Size_KB': round(size_kb, 1), 'Rows': n_rows})

df_sizes = pd.DataFrame(records)
print(df_sizes.to_string(index=False))

# ── 2) Plot único con escala log ──────────────────────────────
sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(9, 6))

palette = {'Original': 'skyblue', 'Distilled': 'lightcoral'}
sns.barplot(data=df_sizes, x='Dataset', y='Size_KB', hue='Type',
            palette=palette, dodge=True, ax=ax)

# ── 3) Escala logarítmica con grid mayor y menor ──────────────
ax.set_yscale('log')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(
    lambda x, _: f"{x:,.0f}" if x >= 1 else f"{x:.2f}"
))
# ax.grid(True, which='major', linestyle='-',  alpha=0.4)
# ax.grid(True, which='minor', linestyle='--', alpha=0.2)
# ax.minorticks_on()

# ── 4) Etiquetas de valor encima de cada barra ────────────────
for container in ax.containers:
    for bar in container:
        h = bar.get_height()
        if h < 1e-6:
            continue
        x = bar.get_x() + bar.get_width() / 2
        ax.text(x, h * 1.08, f"{h:,.0f}",
                ha='center', va='bottom', fontsize=8.5)

# ── 5) Leyenda y etiquetas ────────────────────────────────────
ax.legend(loc='upper left', fontsize=10, title='Type', title_fontsize=10)
ax.set_ylabel("Size (KB) — log scale", fontsize=11)
ax.set_xlabel("Dataset", fontsize=11)
ax.set_title("Dataset size comparison – Original vs Distilled (IPC = 10, k-means)",
             fontsize=12)

plt.tight_layout()
plt.savefig('results/efficiency_figures/comparacion_tamanos_conjuntos_destilados.png',
            dpi=300, bbox_inches='tight')
plt.show()

## Shopper Dataset

In [ ]:
df_only_coreset_shoppers = pd.read_csv('results/checkpoint/checkpoint_shoppers_Doriginal_F/metrics.csv')

df_full_shoppers = df_only_coreset_shoppers[df_only_coreset_shoppers['method'] == 'Full-data']
df_latent_shoppers = pd.read_csv('results/checkpoint/checkpoint_shoppers_Dlatent_F/metrics.csv')
df_latent_ft_shoppers = pd.read_csv('results/checkpoint/checkpoint_shoppers_Dlatent_ft_F/metrics.csv')

In [ ]:
# Información de la evaluación con todos los datos
df_full_shoppers[['model','method','test_balanced','test_roc_auc']]

### Table calculation with IPC values

In [ ]:
tabla = summarize_two_metrics(
    ipc=10,
    #metric='test_balanced',
    df_only_coreset=df_only_coreset_shoppers,
    df_latent=df_latent_shoppers,
    df_tdcoler=pd.DataFrame(),  # No TDColer para shoppers
    df_latent_ft=None,
    df_mlp = df_mlp_shoppers
)
tabla

In [ ]:
tabla = summarize_two_metrics(
    ipc=50,
    #metric='test_balanced',
    df_only_coreset=df_only_coreset_shoppers,
    df_latent=df_latent_shoppers,
    df_tdcoler=pd.DataFrame(),  # No TDColer para shoppers
    df_latent_ft=None
)
tabla

In [ ]:
tabla = summarize_two_metrics(
    ipc=100,
    #metric='test_balanced',
    df_only_coreset=df_only_coreset_shoppers,
    df_latent=df_latent_shoppers,
    df_tdcoler=pd.DataFrame(),  # No TDColer para shoppers
    df_latent_ft=None
)
tabla

### Efficiency metrics calculation for shopper

In [ ]:
methods = ['k-means','ag']
models = ['logreg','mlp','naive_bayes','xgb','svc','rf']

compare_columns = df_full_shoppers.columns

plot_metric_grid_all(
    df_full=df_full_shoppers[compare_columns],
    df_only_coreset=df_only_coreset_shoppers[compare_columns],
    df_latent=df_latent_shoppers[compare_columns],
    df_latent_ft=df_latent_ft_shoppers[compare_columns],
    #df_TDColer=df_tdcoler_latent_default,
    #space='original',      
    df_MLP=df_mlp_shoppers, 
    metric='test_balanced',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='balanced accuracy',
    name_savefig='results/curve_figures/shopper_KM_AG_balanced_accuracy.png'
)

In [ ]:
methods = ['k-means','ag']
models =['logreg','mlp','naive_bayes','xgb','svc','rf']
compare_columns = df_full_shoppers.columns

plot_metric_grid_all(
    df_full=df_full_shoppers[compare_columns],
    df_only_coreset=df_only_coreset_shoppers[compare_columns],
    df_latent=df_latent_shoppers[compare_columns],
    df_latent_ft=df_latent_ft_shoppers[compare_columns],
    #df_TDColer=df_tdcoler_latent_default,
    #space='original',      
    df_MLP=df_mlp_shoppers, 
    metric='test_roc_auc',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='ROC AUC',
    name_savefig='metrics_figs/shoppers_KM_AG_roc_auc.png'
)

In [ ]:
methods = ['lc','k-center']
models = ['logreg','mlp','naive_bayes','xgb','svc','rf']
compare_columns = df_full_shoppers.columns

plot_metric_grid_all(
    df_full=df_full_shoppers[compare_columns],
    df_only_coreset=df_only_coreset_shoppers[compare_columns],
    df_latent=df_latent_shoppers[compare_columns],
    df_latent_ft=df_latent_ft_shoppers[compare_columns],
    #df_TDColer=df_tdcoler_latent_shoppers,
    #space='original',      
    metric='test_balanced',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='balanced accuracy',
    name_savefig='metrics_figs/shoppers_LC_KCENTERS_balanced_accuracy.png'
)

In [ ]:
methods = ['lc','k-center']
models = ['knn','logreg','mlp','naive_bayes','xgb','svc','rf']
compare_columns = df_full_shoppers.columns

plot_metric_grid_all(
    df_full=df_full_shoppers[compare_columns],
    df_only_coreset=df_only_coreset_shoppers[compare_columns],
    df_latent=df_latent_shoppers[compare_columns],
    df_latent_ft=df_latent_ft_shoppers[compare_columns],
    #df_TDColer=df_tdcoler_latent_shoppers,
    #space='original',      
    metric='test_roc_auc',
    methods=methods,
    models=models,
    x_label='IPC',
    y_label='ROC AUC',
    name_savefig='metrics_figs/shoppers_LC_KCENTERS_roc_auc.png'
)

## Ablation Analysis

In [ ]:
"""
Ablation Heatmap: Δ Balanced Accuracy (CorVAE vs MLP-VAE)
- Visuales premium (Sistema de capas para evitar cruces en celdas NaN)
- Escala dinámica fija ±0.30 con colores modernos
- N/A marcado con bloque violeta sólido
- Título genérico (la descripción se pondrá en el caption del paper)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle

# ── Configuración ──────────────────────────────────────────────────────────────
DATASETS = {
    'Adult': {
        'transformer': 'results/checkpoint/checkpoint_adult_Dlatent_G/metrics.csv',
        'mlp':         'checkpoint_adult_MLP_Dlatent/metrics.csv',
    },
    'Default': {
        'transformer': 'results/checkpoint/checkpoint_default_Dlatent_G/metrics.csv',
        'mlp':         'checkpoint_default_MLP_Dlatent/metrics.csv',
    },
    'Shoppers': {
        'transformer': 'results/checkpoint/checkpoint_shoppers_Dlatent_F/metrics.csv',
        'mlp':         'checkpoint_shoppers_MLP_Dlatent/metrics.csv',
    },
    'Paysim': {
        'transformer': 'checkpoint_paysim_Dlatent_I/metrics.csv',
        'mlp':         'checkpoint_paysim_MLP_Dlatent/metrics.csv',
    },
}

# Combinaciones que no completaron por restricciones computacionales
COMPUTATIONAL_NA = {
    ('Paysim', 'svc'),
}

SPACE = 'latent'
METHOD = 'k-means'
METRIC = 'test_balanced'
IPC_LIST = [10, 50, 100]

# Mapeo de nombres de modelos (KNN eliminado)
MODEL_NAMES = {
    'xgb':         'XGBoost',
    'rf':          'RF',
    'svc':         'SVC',
    'logreg':      'LogReg',
    'mlp':         'MLP',
    'naive_bayes': 'NB',
}

# Orden deseado para el eje Y (KNN eliminado)
MODEL_ORDER = ['xgb', 'rf', 'svc', 'logreg', 'mlp', 'naive_bayes']

# ── Funciones Auxiliares ───────────────────────────────────────────────────────
def get_stats_metric(csv_path, space, method, ipc):
    if not Path(csv_path).exists():
        return pd.DataFrame()

    df = pd.read_csv(csv_path)
    mask = (df['space'] == space) & (df['method'] == method) & (df['IPC'] == ipc)
    if not mask.any():
        return pd.DataFrame()

    return df.loc[mask].groupby('model')[METRIC].agg(['mean', 'std', 'count'])


def build_matrices(ipc):
    color_matrix = pd.DataFrame(index=MODEL_ORDER, columns=DATASETS.keys(), dtype=float)
    annot_matrix = pd.DataFrame(index=MODEL_ORDER, columns=DATASETS.keys(), dtype=str)
    cell_type    = pd.DataFrame(index=MODEL_ORDER, columns=DATASETS.keys(), dtype=str)

    for dataset_name, paths in DATASETS.items():
        stats_tf  = get_stats_metric(paths['transformer'], SPACE, METHOD, ipc)
        stats_mlp = get_stats_metric(paths['mlp'],         SPACE, METHOD, ipc)

        for model in MODEL_ORDER:
            # Caso 1: Restricción Computacional
            if (dataset_name, model) in COMPUTATIONAL_NA:
                color_matrix.loc[model, dataset_name] = np.nan
                annot_matrix.loc[model, dataset_name] = 'N/A'
                cell_type.loc[model, dataset_name]    = 'na_computational'
                continue

            # Caso 2: Datos disponibles
            if model in stats_tf.index and model in stats_mlp.index:
                m1, s1, n1 = stats_tf.loc[model,  ['mean', 'std', 'count']]
                m2, s2, n2 = stats_mlp.loc[model, ['mean', 'std', 'count']]

                delta = m1 - m2
                s1_safe, s2_safe = max(s1, 1e-9), max(s2, 1e-9)
                
                _, p_val = stats.ttest_ind_from_stats(m1, s1_safe, n1, m2, s2_safe, n2, equal_var=False)

                if p_val < 0.05:
                    color_matrix.loc[model, dataset_name] = delta
                    annot_matrix.loc[model, dataset_name] = f'{delta:+.3f}*'
                    cell_type.loc[model, dataset_name]    = 'significant'
                else:
                    color_matrix.loc[model, dataset_name] = np.nan
                    annot_matrix.loc[model, dataset_name] = f'{delta:+.3f}'
                    cell_type.loc[model, dataset_name]    = 'non_significant'
            else:
                color_matrix.loc[model, dataset_name] = np.nan
                annot_matrix.loc[model, dataset_name] = ''
                cell_type.loc[model, dataset_name]    = 'missing'

    color_matrix.index = [MODEL_NAMES.get(m, m) for m in color_matrix.index]
    annot_matrix.index = [MODEL_NAMES.get(m, m) for m in annot_matrix.index]
    cell_type.index    = [MODEL_NAMES.get(m, m) for m in cell_type.index]

    return color_matrix, annot_matrix, cell_type


# ── Generar Gráfico ────────────────────────────────────────────────────────────
# Usamos una fuente general limpia
sns.set_theme(style="white", font_scale=1.1)

fig, axes = plt.subplots(1, len(IPC_LIST), figsize=(5.5 * len(IPC_LIST), 6), sharey=True)

# Paleta premium (Azul y Rojo vibrante, centro neutro)
cmap_sig = sns.diverging_palette(10, 250, s=85, l=45, as_cmap=True)
cmap_base = mcolors.ListedColormap(['#EDEDED']) # Gris claro liso para el fondo
COLOR_NA = '#5E35B1' # Violeta fuerte y elegante

VMAX_VAL  =  0.30
VMIN_VAL  = -0.30

for i, ipc in enumerate(IPC_LIST):
    ax = axes[i]
    color_matrix, annot_matrix, cell_type = build_matrices(ipc)

    # 1. Capa Base: Matriz vacía para dibujar un fondo gris impecable y cuadrículas blancas
    sns.heatmap(
        np.zeros(color_matrix.shape),
        ax=ax, cmap=cmap_base, cbar=False, annot=False,
        linewidths=1.5, linecolor='white'
    )

    # 2. Capa Significativa: Sobreponemos solo las celdas coloreadas
    mask_sig = (cell_type != 'significant')
    sns.heatmap(
        color_matrix.astype(float),
        ax=ax, cmap=cmap_sig, center=0, vmin=VMIN_VAL, vmax=VMAX_VAL,
        mask=mask_sig, annot=False, 
        cbar=(i == len(IPC_LIST) - 1), 
        linewidths=1.5, linecolor='white',
        cbar_kws={'label': 'Δ Balanced Accuracy\n(CorVAE − MLP-VAE)'} if i == len(IPC_LIST) - 1 else None
    )

    # 3. Capa de Texto y Parches (N/A)
    model_labels   = color_matrix.index.tolist()
    dataset_labels = color_matrix.columns.tolist()

    for y, model in enumerate(model_labels):
        for x, dataset in enumerate(dataset_labels):
            ctype = cell_type.loc[model, dataset]
            text  = annot_matrix.loc[model, dataset]

            if ctype == 'significant':
                delta_val = color_matrix.loc[model, dataset]
                # Contraste dinámico: blanco para colores oscuros, gris muy oscuro para los claros
                txt_color = 'white' if abs(delta_val) > 0.14 else '#1A1A1A'
                ax.text(x + 0.5, y + 0.5, text, ha='center', va='center',
                        fontsize=11, color=txt_color)

            elif ctype == 'non_significant':
                # Valor liso sobre el fondo gris, sin cruces
                ax.text(x + 0.5, y + 0.5, text, ha='center', va='center',
                        fontsize=10.5, color='#666666')

            elif ctype == 'na_computational':
                # Dibujar bloque violeta manual perfecto
                rect = Rectangle((x, y), 1, 1, fill=True, facecolor=COLOR_NA, edgecolor='white', lw=1.5)
                ax.add_patch(rect)
                ax.text(x + 0.5, y + 0.5, text, ha='center', va='center',
                        fontsize=10, color='white')

    # Estilizado final del panel
    ax.set_title(f'IPC = {ipc}', fontsize=15, pad=12)
    ax.set_xlabel('Dataset', fontsize=13)
    if i == 0:
        ax.set_ylabel('Classifier', fontsize=13)
    
    ax.tick_params(axis='y', rotation=0, labelsize=12)
    ax.tick_params(axis='x', labelsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# ── Título General ─────────────────────────────────────────────────────────────
fig.suptitle(
    'CorVAE vs. MLP-VAE Baseline',
    fontsize=18,
)

plt.tight_layout()

# ── Guardar ────────────────────────────────────────────────────────────────────
save_path = 'results/curve_figures/ablation_heatmap_premium.png'
Path(save_path).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✅ Figura premium guardada en: {save_path}")
plt.show()

## Explainability of latent space efficiency to improve the distillation of coreset techniques

In [ ]:
# Carga de datos para visualización 

from utils import preprocessing, split_train_test_custom, read_json_config
import os

metadata_path = 'data/shoppers/metadata.json'

config = read_json_config(metadata_path)
base_dir = os.path.dirname(metadata_path)

STATIC_SEED = 0
SEED = 0

# Se utiliza un test de 0.1 porque luego se aplicar validacion cruzada que ya incluye validacion
X_train_init, y_train_init, X_test_init, y_test_init = split_train_test_custom(config, base_dir, test_size=0.1, random_state=STATIC_SEED)

set_data = (X_train_init, y_train_init, X_test_init, y_test_init)

(
    X_train_pre, X_test_pre,
    y_train_pre, y_test_pre,
    num_scaler,  cat_encoder,
    label_encoder
                    ) = preprocessing(config, X_train_init, y_train_init, X_test_init, y_test_init, encoding='one-hot', concat=True, random_state=SEED)



In [ ]:
# Calculo del coeficiente de silueta para analizar la cohesion de los clusters
z_root_path = 'results/checkpoint/checkpoint_shoppers_Dlatent_F/vae'
plot_silhouette_vs_ipc_all_methods(
X_train_pre,  y_train_pre,
    ipc_values = list(range(10, 101, 10)),
    seeds      = [5, 6, 7, 8, 9],
    model      = None,
    flat       = True,
    figname    = 'results/explainability_latent_effect/silhouette_analysis.png',
    z_root_path = z_root_path
)

In [ ]:
# Visualización de t-SNE para comparar el espacio original y el latente

from sklearn.manifold import TSNE

X_original, y = X_train_pre,  y_train_pre

# — Ajusta estas rutas/variables —
seed = 5
ipc = 1000
flat = True
Z = load_latent(seed, flat, z_root_path)

# Parámetros
seed = 42
palette = {0: 'blue', 1: 'red'}  # ajusta si tus clases son otras

# Calcula t-SNE en cada espacio
tsne = TSNE(n_components=2, random_state=seed)
emb_o = tsne.fit_transform(X_original)
emb_l = tsne.fit_transform(Z)

# Prepara figura con 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Estilo sencillo: sin grilla
sns.set_style('white')

# Original
sns.scatterplot(
    x=emb_o[:, 0], y=emb_o[:, 1],
    hue=y, palette=palette, legend=False,
    ax=axes[0], alpha=0.4, s=20,
)
axes[0].set_title('Original')
axes[0].grid(False)

# Latent
sns.scatterplot(
    x=emb_l[:, 0], y=emb_l[:, 1],
    hue=y, palette=palette, legend=False,
    ax=axes[1], alpha=0.4, s=20,
)
axes[1].set_title('Latent')
axes[1].grid(False)

plt.tight_layout()
plt.savefig('results/explainability_latent_effect/tsne_original_vs_latente_simple.png')
plt.show()

In [ ]:
# Observar visualmente las muestras destiladas en el espacio original y latente para KMEANS
hyperparams_vae_path = 'tune_vae/tune_vae_AAC/tune_shoppers/best_hyperparams.json'

X_original, y = X_train_pre,  y_train_pre
Z = load_latent(5, False, z_root_path)

plot_kmeans_distill_and_reconstruct_tsne_seaborn(
    X_original=X_original,
    Z=Z,
    y=y,
    ipc=30,
    seed=5,
    metadata_path=metadata_path,
    vae_dir='results/checkpoint/checkpoint_shoppers_Dlatent_F/vae/',
    device='cuda',
    num_scaler=num_scaler,
    cat_encoder=cat_encoder,
    label_encoder=label_encoder,
    concat_label=False,
    hyperparams_vae_path=hyperparams_vae_path,
    figname='results/explainability_latent_effect/tsne_kmeans_distillation.png',
    method='KM'
)

In [ ]:
# Observar visualmente las muestras destiladas en el espacio original y latente para Agglomerative Clustering

plot_kmeans_distill_and_reconstruct_tsne_seaborn(
    X_original=X_original,
    Z=Z,
    y=y,
    ipc=30,
    seed=5,
    metadata_path=metadata_path,
    vae_dir='results/checkpoint/checkpoint_shoppers_Dlatent_F/vae/',
    device='cuda',
    num_scaler=num_scaler,
    cat_encoder=cat_encoder,
    label_encoder=label_encoder,
    concat_label=False,
    hyperparams_vae_path=hyperparams_vae_path,
    figname='results/explainability_latent_effect/tsne_AG_distillation.png',
    method='AG'
)

In [ ]:
# Observar visualmente las muestras destiladas en el espacio original y latente para least confidence

plot_kmeans_distill_and_reconstruct_tsne_seaborn(
    X_original=X_original,
    Z=Z,
    y=y,
    ipc=30,
    seed=5,
    metadata_path=metadata_path,
    vae_dir='results/checkpoint/checkpoint_shoppers_Dlatent_F/vae/',
    device='cuda',
    num_scaler=num_scaler,
    cat_encoder=cat_encoder,
    label_encoder=label_encoder,
    concat_label=False,
    hyperparams_vae_path=hyperparams_vae_path,
    figname='results/explainability_latent_effect/tsne_LC_distillation.png',
    method='LC'
)

In [ ]:
# Observar visualmente las muestras destiladas en el espacio original y latente para K-Centers

plot_kmeans_distill_and_reconstruct_tsne_seaborn(
    X_original=X_original,
    Z=Z,
    y=y,
    ipc=30,
    seed=5,
    metadata_path=metadata_path,
    vae_dir='results/checkpoint/checkpoint_shoppers_Dlatent_F/vae/',
    device='cuda',
    num_scaler=num_scaler,
    cat_encoder=cat_encoder,
    label_encoder=label_encoder,
    concat_label=False,
    hyperparams_vae_path=hyperparams_vae_path,
    figname='results/explainability_latent_effect/tsne_KC_distillation.png',
    method='K-Centers'
)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib
try:
    get_ipython()
except NameError:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

BASELINE_HYPERPARAMS = {
    'max_beta':      0.007715720500892705,
    'min_beta':      0.00014144050146933074,
    'lambda_':       0.9,
    'lr_pretrain':   0.005105439988316641,
    'wd_pretrain':   0,
    'd_token':       8,
    'token_bias':    True,
    'n_head':        2,
    'factor':        16,
    'num_layers':    1,
    'early_stop_counter_pretrain': 30,
    'early_stop_counter_ft': 30,
}

SENSITIVITY_SPACE = {
    'max_beta':    [1e-4, 1e-3, 5e-3, 0.0077, 0.01, 0.05, 0.1],
    'min_beta':    [1e-7, 1e-6, 1e-5, 0.00014, 1e-3],
    'num_layers':  [1, 2, 3, 4],
    'n_head':      [1, 2, 4, 8],
    'd_token':     [4, 8, 16, 32],
    'factor':      [4, 8, 16, 32, 64],
    'lr_pretrain': [1e-4, 5e-4, 1e-3, 0.005, 0.01],
    'lambda_':     [0.1, 0.3, 0.5, 0.7, 0.9],
}

PARAM_LABELS = {
    'max_beta':    r'$\beta_{max}$',
    'min_beta':    r'$\beta_{min}$',
    'num_layers':  'Transformer Layers',
    'n_head':      'Attention Heads',
    'd_token':     'Token Dimension ($d$)',
    'factor':      'FFN Factor',
    'lr_pretrain': 'Learning Rate',
    'lambda_':     r'$\lambda$ (Annealing)',
}


def plot_sensitivity(results_df, output_dir, dataset='shoppers'):
    params    = results_df['param_name'].unique()
    n_params  = len(params)
    n_cols    = (n_params + 1) // 2
    fig, axes = plt.subplots(2, n_cols, figsize=(5 * n_cols, 10))
    axes      = axes.flatten()

    for idx, param in enumerate(params):
        ax     = axes[idx]
        subset = results_df[results_df['param_name'] == param].sort_values('param_value')

        ax.plot(range(len(subset)), subset['roc_auc_recon'].values,
                'o-', color='#2196F3', linewidth=2, markersize=8, label='ROC-AUC (Recon)')
        ax.plot(range(len(subset)), subset['balanced_acc_recon'].values,
                's--', color='#FF9800', linewidth=2, markersize=7, label='Bal. Acc (Recon)')

        baseline_val = BASELINE_HYPERPARAMS.get(param)
        if baseline_val is not None:
            vals_list   = subset['param_value'].tolist()
            closest_idx = min(range(len(vals_list)),
                              key=lambda i: abs(float(vals_list[i]) - float(baseline_val)))
            ax.axvline(x=closest_idx, color='red', linestyle=':', alpha=0.7, label='Optimal')

        ax.set_xticks(range(len(subset)))
        x_labels = subset['param_value'].values
        if param in ('max_beta', 'min_beta', 'lr_pretrain'):
            x_labels = [f'{float(v):.1e}' for v in x_labels]
        else:
            x_labels = [str(v) for v in x_labels]
        ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
        ax.set_title(PARAM_LABELS.get(param, param), fontsize=13)
        ax.set_ylabel('Score', fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=8, loc='lower left')
        ax.grid(True, alpha=0.3)

    for idx in range(n_params, len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(f'Hyperparameter Sensitivity Analysis — CorVAE ({dataset.capitalize()} Dataset)',
                 fontsize=15, y=1.02)
    plt.tight_layout()

    os.makedirs(output_dir, exist_ok=True)
    plot_path = os.path.join(output_dir, 'sensitivity_plots.png')
    fig.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"📊 Plot saved: {plot_path}")
    plt.show()

In [ ]:
# 1) Cargar el CSV con los resultados
results_df = pd.read_csv('tune_vae/sensitivity_shoppers/sensitivity_results.csv')

# 2) Filtrar solo los experimentos exitosos
valid_results = results_df[results_df['status'] == 'ok']

# 3) Llamar la función
plot_sensitivity(
    results_df=valid_results,
    output_dir='tune_vae/sensitivity_shoppers',
    dataset='shoppers'
)